In [1]:
# helper for columns
def require_columns(df, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

In [9]:
from pathlib import Path
import pandas as pd

IN_CSV = "/Users/mariaworkman/fashion/fashion-neutrality/data/samples/sample_2000_2023_100peryear.csv"

IMG_DIR = Path("/Users/mariaworkman/fashion/fashion-neutrality/data/images/sample_2000_2023_100peryear")

df = pd.read_csv(IN_CSV)

# build local_path from filename
df["local_path"] = df["filename"].apply(lambda f: str(IMG_DIR / f))

# keep only rows where the file actually exists on disk
df["download_ok"] = df["local_path"].apply(lambda p: Path(p).exists())
df = df[df["download_ok"]].copy()

# quick peek
print(df.shape)
df[["year", "designer", "season", "local_path"]].head()

(2399, 13)


,year,designer,season,local_path
0,2000,Rebecca Danenberg,Spring,/Users/mariaworkman/fashion/fashion-neutrality...
1,2000,Miu Miu,Spring,/Users/mariaworkman/fashion/fashion-neutrality...
2,2000,Halston,Fall,/Users/mariaworkman/fashion/fashion-neutrality...
3,2000,Helmut Lang,Fall,/Users/mariaworkman/fashion/fashion-neutrality...
4,2000,Loewe,Spring,/Users/mariaworkman/fashion/fashion-neutrality...


In [10]:
# ensuring we didnt drop a ton of rows 
df_raw = pd.read_csv(IN_CSV)
print("raw rows:", len(df_raw))
print("kept rows:", len(df))
print("kept %:", len(df)/len(df_raw))

raw rows: 2400
kept rows: 2399
kept %: 0.9995833333333334


In [11]:
# ensuring jpg formats 
from pathlib import Path

ext_counts = df["local_path"].apply(lambda p: Path(p).suffix.lower()).value_counts()
print(ext_counts.head(10))

local_path
.jpg    2399
Name: count, dtype: int64


In [12]:
# more filtering
from PIL import Image

bad = []
for p in df["local_path"].sample(min(50, len(df)), random_state=0):
    try:
        with Image.open(p) as im:
            im.verify()
    except Exception as e:
        bad.append((p, str(e)))

print("bad in sample:", len(bad))
bad[:3]

bad in sample: 0


[]

In [13]:
# checking duplicates 
print("duplicate filenames:", df["filename"].duplicated().sum())
print("duplicate local_paths:", df["local_path"].duplicated().sum())

duplicate filenames: 0
duplicate local_paths: 0


In [14]:
# designer dupes

print("unique designers:", df["designer"].nunique())
print(df["designer"].value_counts().head(10))


unique designers: 747
designer
Giorgio Armani        28
Valentino             27
Jean Paul Gaultier    23
Louis Vuitton         23
Dolce & Gabbana       22
Emporio Armani        22
Christian Dior        21
Burberry              21
Chanel                21
Elie Saab             20
Name: count, dtype: int64


In [15]:
# creating test for pipeline we'll use eventually 

df_test = df.sample(10, random_state=0).copy()